# Real GDP (RGDP) ranking CIs

Same procedure as `UNEMP_CI` / `NGDP_CI`, but for the SPF real-GDP panel:
- **Forecasts**: `SPFmicrodata.xlsx`, sheet `RGDP` (RGDP1..RGDP6) — quarterly real-GDP level forecasts.
- **Realizations**: `ROUTPUTQvQd.xlsx` — quarterly RTDSM vintage matrix with prefix `ROUTPUT`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from rankci import (
    rank_ci_stepwise_pairwise,
    rank_ci_marginal_pairwise,
    rank_confidence_intervals_simulation_pairwise,
    rank_ci_stepwise_simulation_pairwise,
    rank_ci_marginal_simulation_pairwise,
    compute_pairwise,
    select_top_forecasters,
    winsorize_panel,
)
from rankci.data.philly import (
    load_spf,
    load_rtdsm,
    compute_errors,
    compute_error_panel,
)
from rankci.data.philly import advance_vintage_col, get_advance_estimate

# Data Loading

In [ ]:
df = load_spf("../../data/philly/SPFmicrodata.xlsx", sheet="RGDP")
print(f"SPF RGDP: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
routput = load_rtdsm("../../data/philly/ROUTPUTQvQd.xlsx", prefix="ROUTPUT", freq="quarterly")
print(f"RTDSM (ROUTPUT): {routput.shape[0]} quarters × {routput.shape[1]} vintages")
routput.head()

# Sanity Checks

In [ ]:
# Verify advance real-GDP estimates for a few known quarters
test_cases = [(1995, 2), (2008, 4), (2020, 2)]
print(f"{'Target Quarter':<20} {'Vintage Col':<14} {'Real GDP (bn chained $)'}")
print("-" * 60)
for y, q in test_cases:
    col = advance_vintage_col(y, q, prefix="ROUTPUT")
    val = get_advance_estimate(y, q, routput)
    print(f"  {y}:Q{q:<16}  {col:<14}  {val:.1f}")

# Forecast Errors

In [ ]:
errors_df = compute_errors(df, routput, indicator="RGDP")

print("Error summary by horizon (bn chained $):")
for h in range(1, 7):
    col = f"error_RGDP{h}"
    if col in errors_df.columns:
        print(f"  {col}: mean={errors_df[col].mean():+.2f}, "
              f"std={errors_df[col].std():.2f}, "
              f"nan%={errors_df[col].isna().mean()*100:.1f}%")

errors_df.head()

In [ ]:
# Squared error panel for RGDP3 (one-quarter-ahead)
X_wide = compute_error_panel(df, routput, indicator="RGDP", horizon=3, metric="squared")
print(f"Panel shape: {X_wide.shape[0]} quarters × {X_wide.shape[1]} forecasters")
X_wide.head()

# Forecaster Selection

In [ ]:
obs_counts = X_wide.notna().sum()
mean_mse = X_wide.mean()

stats = pd.DataFrame({
    "obs_count": obs_counts,
    "mean_mse":  mean_mse,
    "rmse":      np.sqrt(mean_mse),
}).sort_values("obs_count", ascending=False)

print(f"Total forecasters: {len(stats)}")
print(f"Forecasters with >= 20 obs: {(stats['obs_count'] >= 20).sum()}")
print(f"Forecasters with >= 50 obs: {(stats['obs_count'] >= 50).sum()}")
stats.head(20)

In [ ]:
N = 8
X_panel = select_top_forecasters(X_wide, N=N, min_obs=20)
print(f"Selected {X_panel.shape[1]} forecasters, {X_panel.shape[0]} quarters")
print(f"Forecaster IDs: {X_panel.columns.tolist()}")

# Rank Confidence Intervals

In [ ]:
X = X_panel.values
population_ids = X_panel.columns.tolist()

out = rank_ci_stepwise_pairwise(X, alpha=0.05, B=5000, seed=42)

results = pd.DataFrame({
    "ID":         population_ids,
    "MSE":        out["theta_hat"].round(4),
    "RMSE":       np.sqrt(out["theta_hat"]).round(4),
    "CI_lower":   out["rank_ci"][:, 0],
    "CI_upper":   out["rank_ci"][:, 1],
}).sort_values("MSE")
results.index = range(1, len(results) + 1)
results.index.name = "Rank"
results

# Worst-Quarter Inspection

In [ ]:
period_mse = X_panel.mean(axis=1)

print(f"Time coverage: {X_panel.index[0]} to {X_panel.index[-1]} ({len(X_panel)} quarters)\n")
print("Top 10 quarters with highest average squared error:")
print(period_mse.nlargest(10))

n = len(X_panel)
plt.figure(figsize=(12, 4))
plt.plot(range(n), period_mse, marker="o", linewidth=1, markersize=3)
plt.xticks(
    ticks=range(0, n, 8),
    labels=[f"{y}Q{q}" for y, q in X_panel.index[::8]],
    rotation=45,
)
plt.ylabel("Avg squared real-GDP error (bn$^2$)")
plt.title(f"Avg squared RGDP3 forecast error across {N} forecasters")
plt.tight_layout()
plt.show()

# Winsorization (variant 2: pairwise differences)

In [ ]:
print("=== Winsorize pairwise differences ===")
print(f"{'winsor_pct':<12} {'max_t':<10} {'mean CI width':<16}")
print("-" * 40)
for winsor_pct in [99, 97, 95, 90]:
    delta_w, se_w, _ = compute_pairwise(X, se_method="nw", winsor_pct=winsor_pct)
    t_vals = (delta_w / se_w)[~np.isnan(se_w)]

    out_w = rank_ci_stepwise_pairwise(
        X, alpha=0.05, B=1000, seed=42, winsor_pct=winsor_pct, verbose=False,
    )
    widths = out_w["rank_ci"][:, 1] - out_w["rank_ci"][:, 0]

    print(f"  {winsor_pct:<10} {t_vals.max():<10.3f} {widths.mean():<16.2f}")

# Marginal Rank Confidence Intervals

Per-forecaster CIs that control coverage marginally (no joint family-wise error control).

In [ ]:
out_marg = rank_ci_marginal_pairwise(X, alpha=0.1, B=5000, seed=42)

results_marg = pd.DataFrame({
    "ID":       population_ids,
    "MSE":      out_marg["theta_hat"].round(4),
    "RMSE":     np.sqrt(out_marg["theta_hat"]).round(4),
    "cv_j":     out_marg["critical_values"].round(3),
    "CI_lower": out_marg["rank_ci"][:, 0],
    "CI_upper": out_marg["rank_ci"][:, 1],
}).sort_values("MSE")
results_marg.index = range(1, len(results_marg) + 1)
results_marg.index.name = "Rank"
results_marg

# Sensitivity to NW bandwidth `L`

Sweep `L ∈ {auto, 1, 2, 4, 8, 20}` for the marginal procedure.

In [ ]:
L_values = [None, 1, 2, 4, 8, 20]

ci_by_L = {}
maxt_by_L = {}
for L in L_values:
    delta, se, _ = compute_pairwise(X, se_method="nw", L=L)
    maxt_by_L[L] = float(np.nanmax(delta / se))
    out_L = rank_ci_marginal_pairwise(X, alpha=0.05, B=2000, seed=42, L=L)
    ci_by_L[L] = out_L["rank_ci"]

order = np.argsort(out_marg["theta_hat"])
ordered_ids = [population_ids[i] for i in order]

table_L = pd.DataFrame(
    {
        f"L={L if L is not None else 'auto'}":
            [f"[{ci_by_L[L][i, 0]},{ci_by_L[L][i, 1]}]" for i in order]
        for L in L_values
    },
    index=ordered_ids,
)
table_L.index.name = "ID"

print("max test stat by L:")
for L, t in maxt_by_L.items():
    label = "auto" if L is None else str(L)
    print(f"  L={label:<5}  max_t = {t:.3f}")
print()
table_L

# MAE-based ranking

In [ ]:
N      = 5
alpha  = 0.2
L      = None

# MAE
X_mae_wide  = compute_error_panel(df, routput, indicator="RGDP", horizon=3, metric="absolute")
X_mae_panel = select_top_forecasters(X_mae_wide, N=N, min_obs=20)
X_mae       = X_mae_panel.values
ids_mae     = X_mae_panel.columns.tolist()

out_mae = rank_ci_marginal_pairwise(X_mae, alpha=alpha, B=5000, seed=42, L=L)

# MSE — same forecasters, same alpha/L, but using squared errors
X_mse_wide  = compute_error_panel(df, routput, indicator="RGDP", horizon=3, metric="squared")
X_mse_panel = select_top_forecasters(X_mse_wide, N=N, min_obs=20)
X_mse       = X_mse_panel.values
ids_mse     = X_mse_panel.columns.tolist()

out_mse = rank_ci_marginal_pairwise(X_mse, alpha=alpha, B=5000, seed=42, L=L)

order = np.argsort(out_mae["theta_hat"])

results_compare = pd.DataFrame({
    "ID":       [ids_mae[i] for i in order],
    "MAE":      out_mae["theta_hat"][order].round(3),
    "MSE":      out_mse["theta_hat"][order].round(4),
    "CI_MAE":   [f"[{l},{u}]" for l, u in out_mae["rank_ci"][order]],
    "CI_MSE":   [f"[{l},{u}]" for l, u in out_mse["rank_ci"][order]],
})
results_compare.index = range(1, len(results_compare) + 1)
results_compare.index.name = "MAE_rank"
results_compare

# τ-best confidence set (Algorithm 3.3)

A forecaster is *included* in the τ-best set iff there is no evidence that — even after exempting any $\tau-1$ others — at least one population outside the exemption is substantially better. We sweep $\tau \in \{1, 2, 3\}$ at $\alpha = 0.2$ and compare Algorithm 3.3 with the naive projection from joint stepwise rank CIs.

In [ ]:
from rankci import tau_best_pairwise, tau_best_from_rank_ci

ALPHA_TAU = 0.2
B_TAU     = 5000
SEED_TAU  = 42

X = X_panel.values
population_ids = X_panel.columns.tolist()

# Joint stepwise rank CIs at the same alpha (basis for the naive projection)
out_rank = rank_ci_stepwise_pairwise(
    X, alpha=ALPHA_TAU, B=B_TAU, seed=SEED_TAU, verbose=False,
)

rows = []
for tau in [1, 2, 3]:
    res = tau_best_pairwise(
        X, tau=tau, alpha=ALPHA_TAU, B=B_TAU, seed=SEED_TAU, verbose=False,
    )
    naive = tau_best_from_rank_ci(out_rank["rank_ci"], tau=tau)
    rows.append({
        "tau":             tau,
        "n_direct":        int(res["n_in_set"]),
        "set_direct":      [population_ids[j] for j, v in enumerate(res["tau_best_set"]) if v],
        "n_naive":         int(naive.sum()),
        "set_naive":       [population_ids[j] for j, v in enumerate(naive) if v],
        "max_test_stat":   round(float(np.nanmax(res["test_stats"])), 3),
    })

tau_sweep = pd.DataFrame(rows).set_index("tau")
print(f"alpha = {ALPHA_TAU}, B = {B_TAU}, p = {len(population_ids)}")
print(f"Forecaster IDs: {population_ids}")
print()
tau_sweep